In [0]:
# https://docs.databricks.com/aws/en/mlflow/

In [0]:
# Notebook: 03_Model_Training
import mlflow
import mlflow.sklearn
from databricks import feature_store

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [0]:
# --- Parameters ---
# These could be passed as widgets or job parameters
n_estimators = 150
max_depth = 10
random_state = 42

mlflow.autolog(
    log_input_examples=True,
    log_model_signatures=True, # Keep signature logging if desired
    log_models=False,  
    silent=True
)

In [0]:
# --- Load Data ---
# Assume 'prepared_df' (with target) and 'fs_table_name' are available or passed
# Option 1: Re-run previous steps if needed (not ideal for jobs)
# Option 2: Load base data and use Feature Store client to join features

# Load base data again (or from Delta Lake) - need primary key and target
# This assumes 01_Data_Ingestion was run or data is accessible
try:
    base_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data2") # Example path
    # OR reload from DB if not saved
    # base_df = spark.read.jdbc(...) # As in notebook 01, select primary_key and target
    base_df = base_df.select("primary_key", "TotalDue") # Need target variable and key
except Exception as e:
    print(f"Could not load base data: {e}")
    dbutils.notebook.exit("Failed to load base data for training.")

In [0]:
fs_table_name = "databricks_us.adventureworks_db.sales_order_features2" # Or get from previous run: dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().get() ...

fs = feature_store.FeatureStoreClient()

# Create Training Set by joining base data (target) with features from Feature Store
# Create a DataFrame with primary keys and timestamps (use OrderDate or current time)
# For simplicity, using just the primary key from our base_df
lookup_keys_df = base_df.select("primary_key")
lookup_keys_df.show(10)

+-----------+
|primary_key|
+-----------+
|      43659|
|      43660|
|      43661|
|      43662|
|      43663|
|      43664|
|      43665|
|      43666|
|      43667|
|      43668|
+-----------+
only showing top 10 rows


In [0]:
display(base_df)

primary_key,TotalDue
43659,23153.234
43660,1457.3289
43661,36865.8
43662,32474.932
43663,472.3108
43664,27510.41
43665,16158.696
43666,5694.8564
43667,6876.3647
43668,40487.723


In [0]:
try:
    training_set = fs.create_training_set(
        df=base_df, # DataFrame containing labels and primary keys
        feature_lookups=[
            feature_store.FeatureLookup(
                table_name=fs_table_name,
                lookup_key="primary_key"
            )
        ],
        label="TotalDue",
        exclude_columns=["primary_key"] # Exclude non-feature/non-label columns
    )
    training_pd = training_set.load_df().toPandas() # Load data into Pandas for scikit-learn
    print("Training set created successfully.")

except Exception as e:
    print(f"Error creating training set from Feature Store: {e}")
    dbutils.notebook.exit("Feature Store training set creation failed.")

Training set created successfully.


In [0]:
artifact_path="model",

In [0]:
%python
# --- Train/Test Split ---
X = training_pd.drop("TotalDue", axis=1)
y = training_pd["TotalDue"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

# --- Model Training with MLflow ---
with mlflow.start_run() as run:
    # Log parameters (Autolog might capture some, but explicit is good too)
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("feature_table", fs_table_name)

    # Train the model
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)
    rf.fit(X_train, y_train)

    # Make predictions
    y_pred = rf.predict(X_test)

    # Log metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Infer model signature
    from mlflow.models.signature import infer_signature
    signature = infer_signature(X_train, y_pred)

    # Log the model using the Feature Store API for better integration
    # This logs the model AND information about the features used
    fs.log_model(
        model=rf,
        artifact_path="model", # Name within MLflow run
        flavor=mlflow.sklearn,
        training_set=training_set, # Pass the training_set object
        signature=signature, # Include the inferred signature
        registered_model_name=None # Register in the next step
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"RMSE: {rmse}")
    print(f"R2: {r2}")
    print("Model logged with Feature Store context.")

# dbutils.notebook.exit(run.info.run_id)

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

2025/04/08 12:35:01 INFO mlflow.tracking._tracking_service.client: 🏃 View run serious-gull-627 at: adb-2035045055759449.9.azuredatabricks.net/ml/experiments/1050600241673767/runs/3d42b40eda6e41a6b0ad4c669570f018.
2025/04/08 12:35:01 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: adb-2035045055759449.9.azuredatabricks.net/ml/experiments/1050600241673767.


Run ID: 3d42b40eda6e41a6b0ad4c669570f018
RMSE: 414.7132572134326
R2: 0.9989973901733561
Model logged with Feature Store context.


In [0]:
# enable autologging
mlflow.sklearn.autolog()

In [0]:
# --- Model Training with MLflow ---
with mlflow.start_run() as run:
    # Train the model
    rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)
    rf.fit(X_train, y_train)

    # Make predictions
    y_pred = rf.predict(X_test)

    print("Model logged with Feature Store context.")

2025/04/08 12:01:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/04/08 12:01:30 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/ml

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

2025/04/08 12:01:34 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/04/08 12:01:34 INFO mlflow.tracking._tracking_service.client: 🏃 View run wistful-wren-907 at: adb-2035045055759449.9.azuredatabricks.net/ml/experime

Model logged with Feature Store context.
